# CGM Preprocessing: Native 5-Minute Resolution

**Replaces the 1-minute-resolution pipeline.** CGMacros' released 1-minute grid is a linear
interpolation over Dexcom's true 5-minute native readings. This notebook extracts the genuine
5-minute-native series per participant, interpolates only across short real gaps, and rebuilds the
sliding-window sample set on top of it.

**Structure**: Sections 1-8 are the main pipeline -- run top to bottom, nothing but the final
train/val/test files depends on anything after this. Section 9 (**Supporting Evidence**) is a
self-contained appendix justifying the choices made above (phase-detection correctness, the three
participant exclusions, the 15-minute interpolation limit, and the resulting image-sample discard
rate) with runnable evidence, not just assertions -- it does not feed back into the saved output.

**Participant exclusions decided going in**: `CGMacros-007`, `CGMacros-018`, `CGMacros-026` are all
excluded; see Section 9.2-9.3 for the evidence. Excluding all three leaves exactly 14 Healthy / 14
Prediabetes / 14 Type 2 Diabetes participants, a *more* balanced cohort than the original 15/16/14.

## 1. Setup

In [1]:
import os
import json
import pickle

import numpy as np
import pandas as pd

DATA_ROOT = '.'  # CGMacros/ and bio.csv are local symlinks alongside this notebook

# -- Pipeline configuration --
NATIVE_STEP_MIN   = 5     # Dexcom's true native sampling interval
WINDOW_MINUTES    = 120   # input window duration (default; 60/120/240 compared later once a
                          # model is selected -- see methodology_experimental_design.tex)
WINDOW_STEPS      = WINDOW_MINUTES // NATIVE_STEP_MIN         # = 24 native points
HORIZON_MINUTES   = [15, 30, 60, 90, 120]
HORIZON_STEPS     = [h // NATIVE_STEP_MIN for h in HORIZON_MINUTES]  # = [3, 6, 12, 18, 24]
STEP_STEPS        = 1     # slide by 1 native point (= 5 minutes) between consecutive samples
MAX_GAP_STEPS      = 3     # interpolate at most 3 consecutive missing native points (~15 min;
                          # see Section 9.4 for why this limit)
IMAGE_LOOKBACK_MIN = 60

EXCLUDE_PIDS = {'CGMacros-007', 'CGMacros-018', 'CGMacros-026'}  # see Section 9.2-9.3

OUT_DIR = os.path.join(DATA_ROOT, 'processed_native5min')
os.makedirs(OUT_DIR, exist_ok=True)
print(f"WINDOW_STEPS={WINDOW_STEPS}  HORIZON_STEPS={HORIZON_STEPS}  "
      f"MAX_GAP_STEPS={MAX_GAP_STEPS}  OUT_DIR={OUT_DIR}")

WINDOW_STEPS=24  HORIZON_STEPS=[3, 6, 12, 18, 24]  MAX_GAP_STEPS=3  OUT_DIR=./processed_native5min


## 2. Phase Detection

In [2]:
def load_full_grid(pid):
    """Dexcom GL reindexed onto the complete, gap-preserving 1-minute timeline spanning this
    participant's monitoring period (first to last non-missing Dexcom value) -- not the raw file's
    own row range, which can extend well before the sensor started or after it stopped for good."""
    df = pd.read_csv(os.path.join(DATA_ROOT, 'CGMacros', pid, f'{pid}.csv'))
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.set_index('Timestamp').sort_index()
    gl = df['Dexcom GL']
    valid = gl.dropna()
    full_idx = pd.date_range(valid.index.min(), valid.index.max(), freq='1min')
    return gl.reindex(full_idx)


def find_breakpoints(gl_1min):
    """Boolean mask: True where the second difference of a gap-preserving 1-minute series is
    non-zero -- the curve's slope changed starting at this row. This flags the row *after* the
    genuine reading (see Section 9.1), not the reading itself."""
    d2 = gl_1min.diff().diff()
    return d2.abs() > 1e-6


def find_real_readings(gl_1min):
    """`find_breakpoints`, shifted back one minute so it marks the genuine Dexcom reading
    itself rather than its one-minute echo in the second difference (Section 9.1)."""
    return find_breakpoints(gl_1min).shift(-1, fill_value=False)


def dominant_phase(gl_1min, min_real=10):
    """The single most common (minute % 5) among this participant's true real-reading
    positions, and what fraction of real readings share it."""
    is_real = find_real_readings(gl_1min)
    if is_real.sum() < min_real:
        return None, 0.0
    counts = (gl_1min.index[is_real].minute % 5).value_counts(normalize=True)
    return int(counts.index[0]), float(counts.iloc[0])

## 3. Extracting the Native 5-Minute Series

In [3]:
def extract_native_5min(gl_1min, phase):
    """Native 5-minute-resolution series aligned to `phase` (minute % 5). Positions with no
    real Dexcom reading remain NaN -- interpolation is a separate, later step."""
    native_index = gl_1min.index[gl_1min.index.minute % 5 == phase]
    native_index = pd.date_range(native_index.min(), native_index.max(),
                                  freq=f'{NATIVE_STEP_MIN}min')
    return gl_1min.reindex(native_index)

## 4. Gap-Limited Linear Interpolation

In [4]:
def interpolate_native(native_series, max_gap_steps=MAX_GAP_STEPS):
    """Linear interpolation across at most `max_gap_steps` consecutive missing native points
    (~15 minutes at the default 5-minute native step); longer gaps are left missing. See Section
    9.4 for why this limit, and Section 9.5 for its measured cost."""
    return native_series.interpolate(method='linear', limit=max_gap_steps,
                                      limit_direction='forward')

## 5. Meal-Image Matching

In [5]:
def get_meal_info(df_raw, t_end, participant_dir, lookback_min=IMAGE_LOOKBACK_MIN):
    """Only matches "tagged" photos (Image path AND Meal Type both present on the same row) --
    "untagged" photos (Image path present, Meal Type absent) are excluded from matching entirely,
    per dataset/meal_analysis.ipynb Section 3-4: untagged photos are overwhelmingly a same-meal
    follow-up shot taken shortly after a tagged photo (median gap well under an hour) with no
    logged nutrition data of their own, and visually show already-consumed food -- carrying no
    information a nearby tagged photo doesn't already provide, so matching against them would let
    a lower-information "after" photo silently displace the correct "before" one whenever the
    after-shot happens to be more recent within the lookback window."""
    lookback_start = t_end - pd.Timedelta(minutes=lookback_min)
    mask = ((df_raw.index >= lookback_start) & (df_raw.index <= t_end)
            & df_raw['Image path'].notna() & df_raw['Meal Type'].notna())
    meal_rows = df_raw[mask]
    if meal_rows.empty:
        return None, 0.0
    last_meal = meal_rows.iloc[-1]
    meal_time = meal_rows.index[-1]
    time_since_meal = (t_end - meal_time).total_seconds() / 60.0
    abs_path = os.path.join(participant_dir, last_meal['Image path'])
    if not os.path.exists(abs_path):
        return None, 0.0
    return abs_path, float(time_since_meal)

## 6. Sliding-Window Sample Construction

In [6]:
def create_samples(native_series, df_raw, pid, participant_dir):
    samples = []
    max_horizon_steps = max(HORIZON_STEPS)
    n = len(native_series)

    for i in range(0, n - WINDOW_STEPS - max_horizon_steps, STEP_STEPS):
        cgm_window = native_series.iloc[i: i + WINDOW_STEPS].values
        if np.isnan(cgm_window).any():
            continue

        t_end = native_series.index[i + WINDOW_STEPS - 1]

        targets = {}
        valid = True
        for h_min, h_steps in zip(HORIZON_MINUTES, HORIZON_STEPS):
            target_idx = i + WINDOW_STEPS - 1 + h_steps
            if target_idx >= n:
                valid = False
                break
            target_val = native_series.iloc[target_idx]
            if np.isnan(target_val):
                valid = False
                break
            targets[h_min] = float(target_val)
        if not valid:
            continue

        image_path, time_since_meal = get_meal_info(df_raw, t_end, participant_dir)

        samples.append({
            'participant_id':  pid,
            'timestamp':       t_end,
            'cgm_sequence':    cgm_window.astype(np.float32),
            'image_path':      image_path,
            'has_image':       image_path is not None,
            'time_since_meal': time_since_meal,
            **{f'target_{h}': targets[h] for h in HORIZON_MINUTES},
        })
    return samples

## 7. Run the Full Pipeline Across All Participants

In [7]:
cgm_root = os.path.join(DATA_ROOT, 'CGMacros')
all_pids = sorted(d for d in os.listdir(cgm_root) if d.startswith('CGMacros-'))
kept_pids = [p for p in all_pids if p not in EXCLUDE_PIDS]

print(f"Total participants: {len(all_pids)}  Excluded: {sorted(EXCLUDE_PIDS)}  "
      f"Kept: {len(kept_pids)}")

all_samples = []
phase_report = []
native_by_pid = {}  # cached for Section 9's supporting evidence -- (raw, interpolated) per pid

for pid in kept_pids:
    pdir = os.path.join(cgm_root, pid)
    gl1 = load_full_grid(pid)
    phase, purity = dominant_phase(gl1)
    phase_report.append({'participant_id': pid, 'phase': phase, 'purity': purity})
    if phase is None or purity < 0.95:
        print(f"  !! {pid}: phase detection failed or unstable (purity={purity:.2f}) -- skipping")
        continue

    native_raw = extract_native_5min(gl1, phase)
    native = interpolate_native(native_raw)
    native_by_pid[pid] = (native_raw, native)

    df_raw = pd.read_csv(os.path.join(pdir, f'{pid}.csv'))
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'])
    df_raw = df_raw.set_index('Timestamp').sort_index()

    all_samples.extend(create_samples(native, df_raw, pid, pdir))

phase_report = pd.DataFrame(phase_report)
print(f"\nAll {len(kept_pids)} kept participants: phase purity range "
      f"[{phase_report['purity'].min():.3f}, {phase_report['purity'].max():.3f}]")
print(f"Total samples: {len(all_samples):,}")
n_img = sum(s['has_image'] for s in all_samples)
print(f"Samples with image: {n_img:,} ({100*n_img/len(all_samples):.1f}%)")

Total participants: 45  Excluded: ['CGMacros-007', 'CGMacros-018', 'CGMacros-026']  Kept: 42



All 42 kept participants: phase purity range [1.000, 1.000]
Total samples: 117,113
Samples with image: 22,512 (19.2%)


## 8. Participant-Level Stratified Split (30 / 6 / 6, Balanced by Glycaemic Group)

In [8]:
bio = pd.read_csv(os.path.join(DATA_ROOT, 'bio.csv'))
bio['participant_id'] = bio['subject'].apply(lambda s: f'CGMacros-{s:03d}')

def glycemic_group(a1c):
    if a1c < 5.7:
        return 'Healthy'
    elif a1c < 6.5:
        return 'Prediabetes'
    return 'Type2Diabetes'

bio['group'] = bio['A1c PDL (Lab)'].apply(glycemic_group)
bio = bio[bio['participant_id'].isin(kept_pids)].set_index('participant_id')

print(bio['group'].value_counts())
assert set(bio['group'].value_counts()) == {14}, "expected exactly 14 per group after exclusions"

# Within each group: sort by the participant's own mean CGM value (from the samples just built),
# then take 2 for test, 2 for val, 10 for train -- spread across the sorted range rather than
# grabbing adjacent extremes.
pid_mean_gl = {}
for s in all_samples:
    pid_mean_gl.setdefault(s['participant_id'], []).append(s['cgm_sequence'].mean())
pid_mean_gl = {p: np.mean(v) for p, v in pid_mean_gl.items()}

train_pids, val_pids, test_pids = set(), set(), set()
for grp in ['Healthy', 'Prediabetes', 'Type2Diabetes']:
    grp_pids = sorted(bio.index[bio['group'] == grp], key=lambda p: pid_mean_gl[p])
    val_pids.update([grp_pids[4], grp_pids[9]])
    test_pids.update([grp_pids[1], grp_pids[12]])
    train_pids.update(p for p in grp_pids if p not in val_pids and p not in test_pids)

splits = {'train': [], 'val': [], 'test': []}
for s in all_samples:
    if s['participant_id'] in train_pids:
        splits['train'].append(s)
    elif s['participant_id'] in val_pids:
        splits['val'].append(s)
    elif s['participant_id'] in test_pids:
        splits['test'].append(s)

for name, pids_ in [('train', train_pids), ('val', val_pids), ('test', test_pids)]:
    grp_counts = bio.loc[list(pids_), 'group'].value_counts().to_dict()
    print(f"{name:5s}: {len(pids_)} participants {grp_counts}, {len(splits[name]):,} samples")

group
Healthy          14
Type2Diabetes    14
Prediabetes      14
Name: count, dtype: int64


train: 30 participants {'Prediabetes': 10, 'Type2Diabetes': 10, 'Healthy': 10}, 83,532 samples
val  : 6 participants {'Type2Diabetes': 2, 'Healthy': 2, 'Prediabetes': 2}, 16,739 samples
test : 6 participants {'Type2Diabetes': 2, 'Healthy': 2, 'Prediabetes': 2}, 16,842 samples


## 9. Normalization Statistics (Training Split Only)

In [9]:
train_samples = splits['train']
seqs = np.concatenate([s['cgm_sequence'] for s in train_samples])
targets = np.array([[s[f'target_{h}'] for h in HORIZON_MINUTES] for s in train_samples]).ravel()
all_vals = np.concatenate([seqs, targets])

cgm_stats = {'cgm_mean': float(all_vals.mean()), 'cgm_std': float(all_vals.std())}
print(f"cgm_mean = {cgm_stats['cgm_mean']:.2f} mg/dL, cgm_std = {cgm_stats['cgm_std']:.2f} mg/dL")

cgm_mean = 140.10 mg/dL, cgm_std = 42.13 mg/dL


## 10. Save Outputs

In [10]:
for name, split_samples in splits.items():
    with open(os.path.join(OUT_DIR, f'{name}.pkl'), 'wb') as f:
        pickle.dump(split_samples, f)

with open(os.path.join(OUT_DIR, 'cgm_stats.json'), 'w') as f:
    json.dump(cgm_stats, f, indent=2)

with open(os.path.join(OUT_DIR, 'split_participants.json'), 'w') as f:
    json.dump({'train': sorted(train_pids), 'val': sorted(val_pids),
               'test': sorted(test_pids), 'excluded': sorted(EXCLUDE_PIDS)}, f, indent=2)

print(f"Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> {OUT_DIR}")

Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> ./processed_native5min


---
# 9. Supporting Evidence

Everything above is the pipeline itself; everything below justifies the choices it makes, with
runnable evidence rather than assertions. Nothing in this section feeds back into the saved
output -- it can be skipped entirely without changing `processed_native5min/`.

## 9.1 The 1-Minute Phase Correction, Verified

If the real reading is at minute $T$, $d_1[T]$ still reflects the slope of the segment *ending* at
$T$, and only $d_1[T{+}1]$ reflects the new segment's slope -- so the non-zero second difference
appears one minute *after* the real reading. Verified directly against `CGMacros-001`'s raw values
below: the first difference is a constant 3.6 through 11:44, becomes a constant 2.8 starting at
11:45 -- the slope genuinely changes at 11:44, but the non-zero $d_2$ only appears at 11:45.

In [11]:
pid_demo = 'CGMacros-001'
gl1_demo = load_full_grid(pid_demo)
seg = gl1_demo.loc['2020-05-01 11:39:00':'2020-05-01 11:53:00']
d1 = seg.diff()
d2 = d1.diff()
print(pd.DataFrame({'gl': seg, 'd1': d1, 'd2': d2}))

phase_demo, purity_demo = dominant_phase(gl1_demo)
native_demo_raw, native_demo = native_by_pid[pid_demo]
print(f"\n{pid_demo}: phase={phase_demo}  purity={purity_demo:.1%}")
print(f"1-minute grid: {len(gl1_demo)} rows, {gl1_demo.notna().sum()} non-null")
print(f"native 5-minute grid: {len(native_demo_raw)} rows, {native_demo_raw.notna().sum()} "
      f"non-null ({native_demo_raw.isna().sum()} genuine gaps, "
      f"{native_demo.isna().sum()} still missing after gap-limited interpolation)")

                        gl   d1            d2
2020-05-01 11:39:00  141.0  NaN           NaN
2020-05-01 11:40:00  144.6  3.6           NaN
2020-05-01 11:41:00  148.2  3.6  0.000000e+00
2020-05-01 11:42:00  151.8  3.6  2.842171e-14
2020-05-01 11:43:00  155.4  3.6 -2.842171e-14
2020-05-01 11:44:00  159.0  3.6  0.000000e+00
2020-05-01 11:45:00  161.8  2.8 -8.000000e-01
2020-05-01 11:46:00  164.6  2.8 -2.842171e-14
2020-05-01 11:47:00  167.4  2.8  2.842171e-14
2020-05-01 11:48:00  170.2  2.8 -2.842171e-14
2020-05-01 11:49:00  173.0  2.8  2.842171e-14
2020-05-01 11:50:00  173.6  0.6 -2.200000e+00
2020-05-01 11:51:00  174.2  0.6  0.000000e+00
2020-05-01 11:52:00  174.8  0.6  2.842171e-14
2020-05-01 11:53:00  175.4  0.6 -2.842171e-14

CGMacros-001: phase=4  purity=100.0%
1-minute grid: 14275 rows, 14265 non-null
native 5-minute grid: 2855 rows, 2853 non-null (2 genuine gaps, 0 still missing after gap-limited interpolation)


## 9.2 `CGMacros-007`: Excluded Because No Stable Phase Exists

Cross-referencing the Dexcom trace against the independently-sampled Libre channel in the same
irregular-interval windows: if the pipeline itself were broken, both channels would look wrong; if
only Dexcom's *device* were producing irregularly-timed readings, Libre in the same windows should
still look like ordinary two-point linear interpolation. Checked across every one of `007`'s 58
irregular-interval regimes (not just a sample).

In [12]:
pid_007 = 'CGMacros-007'
df_007 = pd.read_csv(os.path.join(DATA_ROOT, 'CGMacros', pid_007, f'{pid_007}.csv'))
df_007['Timestamp'] = pd.to_datetime(df_007['Timestamp'])
df_007 = df_007.set_index('Timestamp').sort_index()
dex_007 = df_007['Dexcom GL'].dropna()
lib_007 = df_007['Libre GL'].dropna()

interval_df = pd.DataFrame({'start': dex_007.index[:-1], 'end': dex_007.index[1:]})
interval_df['interval_minutes'] = ((interval_df['end'] - interval_df['start'])
                                    .dt.total_seconds() / 60).astype(int)
interval_df['regime_id'] = (interval_df['interval_minutes']
                             != interval_df['interval_minutes'].shift()).cumsum()
regimes = (interval_df.groupby('regime_id')
           .agg(start=('start', 'first'), end=('end', 'last'),
                interval_minutes=('interval_minutes', 'first'))
           .reset_index(drop=True))
abnormal = regimes[regimes['interval_minutes'] > 1].copy()

def is_linear(series, atol=1e-6):
    d = series.diff().dropna()
    return None if len(d) < 2 else bool(np.allclose(d.values, d.iloc[0], atol=atol))

results = []
for _, row in abnormal.iterrows():
    dex_seg = dex_007.loc[row['start']:row['end']]
    lib_seg = lib_007.loc[row['start']:row['end']].dropna()
    results.append({'dexcom_linear': is_linear(dex_seg), 'libre_linear': is_linear(lib_seg)})
res_007 = pd.DataFrame(results)
testable = res_007.dropna(subset=['dexcom_linear'])

print(f"Irregular-interval regimes checked: {len(res_007)}")
print(f"Dexcom non-linear (genuine, varying trajectory): "
      f"{(~testable['dexcom_linear'].astype(bool)).sum()} / {len(testable)}")
print(f"Libre linear (matches standard interpolation everywhere else in the dataset): "
      f"{testable['libre_linear'].astype(bool).sum()} / {len(testable)}")
print("-> the file-construction pipeline works normally for this participant (Libre proves it); "
      "the irregularity is specific to Dexcom device/connectivity for this participant.")

Irregular-interval regimes checked: 58
Dexcom non-linear (genuine, varying trajectory): 58 / 58
Libre linear (matches standard interpolation everywhere else in the dataset): 58 / 58
-> the file-construction pipeline works normally for this participant (Libre proves it); the irregularity is specific to Dexcom device/connectivity for this participant.


## 9.3 `CGMacros-018` / `CGMacros-026`: Recoverable, Not Discarded Lightly

Per-day dominant phase. Both participants show one single, cleanly-isolated shift consistent with a
mid-study sensor change -- evidence that excluding them (Section 1) is a simplicity choice, not a
sign the data itself is unrecoverable.

In [13]:
def per_day_phase(gl_1min):
    is_real = find_real_readings(gl_1min)
    real_times = gl_1min.index[is_real]
    days = pd.Series(real_times.date, index=real_times)
    phases = pd.Series(real_times.minute % 5, index=real_times)
    daily_phase = phases.groupby(days).agg(lambda x: x.value_counts().idxmax())
    daily_purity = phases.groupby(days).agg(lambda x: x.value_counts(normalize=True).iloc[0])
    return pd.DataFrame({'phase': daily_phase, 'purity': daily_purity})


for pid_check in ['CGMacros-018', 'CGMacros-026']:
    gl1_check = load_full_grid(pid_check)
    daily_df = per_day_phase(gl1_check)
    print(f"\n{pid_check}:")
    print(daily_df.to_string())


CGMacros-018:
            phase  purity
2024-01-28      4     1.0
2024-02-07      1     1.0
2024-02-08      1     1.0
2024-02-09      1     1.0
2024-02-10      1     1.0
2024-02-11      1     1.0
2024-02-12      1     1.0
2024-02-13      1     1.0
2024-02-14      1     1.0
2024-02-15      1     1.0
2024-02-16      1     1.0

CGMacros-026:
            phase  purity
2021-03-26      1     1.0
2021-03-27      1     1.0
2021-03-28      1     1.0
2021-03-29      1     1.0
2021-03-30      1     1.0
2021-04-01      2     1.0
2021-04-02      2     1.0
2021-04-03      2     1.0
2021-04-04      2     1.0
2021-04-05      2     1.0
2021-04-06      2     1.0


## 9.4 Why a 15-Minute Interpolation Limit

The length distribution of genuine gaps in the native series, across all 42 retained participants,
computed *before* any interpolation is applied.

In [14]:
gap_events = []
for pid, (native_raw, _) in native_by_pid.items():
    is_na = native_raw.isna().values
    i, n = 0, len(is_na)
    while i < n:
        if is_na[i]:
            j = i
            while j < n and is_na[j]:
                j += 1
            gap_events.append({'participant_id': pid, 'gap_steps': j - i, 'gap_minutes': (j - i) * 5})
            i = j
        else:
            i += 1

gaps = pd.DataFrame(gap_events)
print(f"{len(gaps)} independent gap events across {len(native_by_pid)} participants")
print(f"mean {gaps.gap_minutes.mean():.1f} min, median {gaps.gap_minutes.median():.1f} min")
print(gaps.gap_minutes.value_counts().sort_index())

fillable = gaps[gaps.gap_steps <= MAX_GAP_STEPS]
toolong = gaps[gaps.gap_steps > MAX_GAP_STEPS]
print(f"\n<= {MAX_GAP_STEPS * NATIVE_STEP_MIN} min (fully filled): {len(fillable)} "
      f"({100*len(fillable)/len(gaps):.1f}%)")
print(f"> {MAX_GAP_STEPS * NATIVE_STEP_MIN} min (left missing): {len(toolong)} "
      f"({100*len(toolong)/len(gaps):.1f}%), mean {toolong.gap_minutes.mean():.1f} min")
print("-> gap lengths are sharply bimodal with nothing observed between the two limits above; "
      "raising the limit into that empty region would recover nothing.")

59 independent gap events across 42 participants
mean 10.0 min, median 5.0 min
gap_minutes
5     47
10     4
15     2
30     1
35     1
40     1
50     1
65     2
Name: count, dtype: int64

<= 15 min (fully filled): 53 (89.8%)
> 15 min (left missing): 6 (10.2%), mean 47.5 min
-> gap lengths are sharply bimodal with nothing observed between the two limits above; raising the limit into that empty region would recover nothing.


## 9.5 The Cost of That Limit, and Its Skew Toward Image-Bearing Windows

Every candidate window across all 42 participants, checked against the same validity rule as
Section 6, split by whether a food photograph falls in its lookback window.

In [15]:
tot_lost_img = tot_kept_img = tot_lost_noimg = tot_kept_noimg = 0
lost_image_records = []

for pid in kept_pids:
    native_raw, native = native_by_pid[pid]
    vals, times = native.values, native.index.values
    n = len(vals)
    max_h = max(HORIZON_STEPS)

    pdir = os.path.join(cgm_root, pid)
    df_raw = pd.read_csv(os.path.join(pdir, f'{pid}.csv'))
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'])
    df_raw = df_raw.set_index('Timestamp').sort_index()
    img_notna = df_raw['Image path'].notna()

    def has_image_near(t_end):
        lookback_start = t_end - pd.Timedelta(minutes=IMAGE_LOOKBACK_MIN)
        return ((df_raw.index >= lookback_start) & (df_raw.index <= t_end) & img_notna).any()

    for i in range(0, n - WINDOW_STEPS - max_h):
        window = vals[i:i + WINDOW_STEPS]
        bad = np.isnan(window).any()
        if not bad:
            for h in HORIZON_STEPS:
                if np.isnan(vals[i + WINDOW_STEPS - 1 + h]):
                    bad = True
                    break
        t_end = pd.Timestamp(times[i + WINDOW_STEPS - 1])
        img = has_image_near(t_end)
        if img:
            if bad:
                tot_lost_img += 1
            else:
                tot_kept_img += 1
        else:
            if bad:
                tot_lost_noimg += 1
            else:
                tot_kept_noimg += 1

total_img, total_noimg = tot_lost_img + tot_kept_img, tot_lost_noimg + tot_kept_noimg
print(f"Image-bearing candidates: {total_img}, discarded {tot_lost_img} "
      f"({100*tot_lost_img/total_img:.3f}%)")
print(f"Non-image candidates: {total_noimg}, discarded {tot_lost_noimg} "
      f"({100*tot_lost_noimg/total_noimg:.3f}%)")
print(f"Relative risk: {(tot_lost_img/total_img)/(tot_lost_noimg/total_noimg):.2f}x")
print("-> small in absolute terms, but a real, documented skew -- plausibly because reopening the "
      "study app to photograph a meal is the same interaction that resumes a lapsed CGM sync.")

Image-bearing candidates: 22674, discarded 92 (0.406%)
Non-image candidates: 94725, discarded 194 (0.205%)
Relative risk: 1.98x
-> small in absolute terms, but a real, documented skew -- plausibly because reopening the study app to photograph a meal is the same interaction that resumes a lapsed CGM sync.
